## <code>data-conduit</code>: Overview and Workflow Guide
**A Python Toolkit for Constructing Standardised, Reusable Data Processing Workflows**

-----

---

### Table of Contents
TODO: Make API/Table of contents go first. 
TODO: Add Introduction section
TODO: New layout should be:
    1. Introduction:
        1.1. What is `data-conduit` 
            - data-conduit is a Python toolkit for converting outputs from multiple data acquistion sources into consistent, unified data structures that can be used for subsequent analysis. This library was originally designed for experimental neuroscience labs working with the Bonsai and HARP ecosystems as part of their workflow, with raw experimental data stored in directories based on the NeuroBlueprint standard. However, most functionalities provided by <code>data-conduit</code> are modular and flexible, and should be compatible with other data acquisition platforms, as well as other data conventions- `data-conduit` is concerned with using a consistent structure to automate data processing workflows, rather than enforcing any one particular format. 
            - explain what this is for, main functionalities are:
                i) automated discovery and reading of files
                ii) processing and conversion of raw data in directories into unified data structures
                iii) Flexible querying utilities to interact with unified datastructures
                iv) temporal alignment of different experimental data 
                v) segmentation of data using querying. 
        1.2. How  does `data-conduit` work?
            - To explain how `data-conduit` works, it is helpful to first explain how data from HARP devices would typically need to be extracted using the `harp-python` API....
            - toolkit therefore exploits the one consistent thing across all experimental sessions, which is the Neuroblueprint directory structure that `data-shuttle` returns from the Bonsai workflow outputs. 
            - data will always reside at .../[device]/[register]/columns. 
        1.3 An Overview of `data-conduit`'s Modules
                - Overview of modules in `data-conduit`
                - TODO: add table of modules and brief description of what they do
    
    2. Using `data-conduit`
        - Made up of 11 modules.
        - Some are primarily concerned with lower-level functions, and users are unlikely to need to use them unless making customised utilities
            - utils
            - timestamps
            - io
            - virtual_arrays
            - harptools
        - Others are 


        
| # | Section | Module(s) |
|---|---------|-----------|
| 1 | [IO — File Discovery & Reading](#1-io) | `io.collect_dfs`, `io.readers` |
| 2 | [DataSource & Subclasses](#2-datasource) | `datasource.DataSource`, `Device`, `FileTypeData` + presets |
| 3 | [MultiSource & MultiDevice](#3-multisource) | `multisource.MultiSource`, `MultiDevice`, `Nosepoke` |
| 4 | [Virtual Arrays & Queries](#4-virtualarrays) | `virtualarrays.construct_data_array`, `construct_lookup_array`, `ulookup` |
| 5 | [TTL Synchronisation](#5-ttlsync) | `ttlsync.extract_ttl_segments`, `TTLSyncModel` |
| 6 | [Global Timebase Mapping](#6-globaltimes) | `globaltimes.create_global_clock`, `index_map_util` |
| 7 | [Segmentation Utilities](#7-segment) | `segment.segment_boolean_series`, `slice_event_windows`, `slice_dataarray_windows` |
| 8 | [Timestamps & Utilities](#8-utilities) | `timestamps`, `harptools`, `utils` |
| 9 | [API Quick-Reference](#9-api) | Full function index |

Part 1: What the Library Does
data-conduit loads experimental data from directory trees, organises it into clean structures, and provides tools for querying, aligning, and segmenting that data.
A typical experiment produces a folder full of device subfolders, each containing data files. The user wants to go from that folder to named, queryable data structures — without manually navigating paths, parsing filenames, or stitching data across devices

---
<a id="9-api"></a>
## 9  API Quick-Reference

### IO (`data_conduit.io`)
| Function | Purpose |
|----------|---------|
| `collect_folders(base_path, folder_prefix)` | List subdirectories with optional prefix filter |
| `collect_dfs(base_path, readers, ...)` | **Master function:** directory → nested dict of DataFrames |
| `read_csv(path, **kwargs)` | Read CSV with sensible defaults |
| `read_json(path, **kwargs)` | Read JSON/JSONL |
| `split_jsonl(path)` | Split JSONL → (metadata_df, trials_df) |
| `split_yaml(path)` | Split YAML → (metadata_df, trials_df) |
| `add_reader(name, fn)` | Register a custom reader |
| `get_reader(name)` | Retrieve a registered reader |

### DataSource (`data_conduit.datasource`)
| Class | Purpose |
|-------|---------|
| `DataSource(experiment_directory_path, readers, datasource_data_arrays, ...)` | Base: directory → dfs_dict + named DataArrays |
| `Device(experiment_directory_path, harp_device_yaml_path, device_type, ...)` | HARP .bin device wrapper |
| `SoundCard(...)` | Preset: SoundCard registers 8, 32, 33, 35 |
| `CameraStart(...)` | Preset: camera start register 78 |
| `Camera0Frames(...)` | Preset: camera frames with optional matching filter |
| `AnalogSync(...)` | Preset: analog sync with optional matching filter |
| `FileTypeData(experiment_directory_path, file_type, ...)` | Flat file (CSV/JSON/YAML) loader |
| `ExperimentEvents(...)` | Preset: CSV events with Value→Event rename |
| `RotationData(...)` | Preset: rotation encoder CSV with angular unit conversion |
| `VideoData(...)` | Preset: VideoData CSV |
| `VisualEnvironment(...)` | Preset: headerless visual environment CSV |
| `RingDebugData(...)` | Preset: YAML ring debug split |

### MultiSource (`data_conduit.multisource`)
| Class | Purpose |
|-------|---------|
| `MultiSource(dfs_dict, virtual_maps, ...)` | Base: dfs_dict + virtual maps → data_arrays + lookup_arrays |
| `MultiDevice(experiment_directory_path, ...)` | HARP device loading + auto virtual-map building |
| `Nosepoke(...)` | Preset: 6 devices × 3 local IDs for nosepoke peripherals |

### Virtual Arrays (`data_conduit.virtualarrays`)
| Function/Class | Purpose |
|----------------|---------|
| `construct_data_array(virtual_map, times, ...)` | Build base xarray DataArray with virtual coords |
| `construct_lookup_array(virtual_map, ...)` | Build 1D lookup array |
| `update_data_array(data_array, virtual_map, dfs_dict, ...)` | Populate DataArray with values from dfs_dict |
| `ulookup(lookup_array, **selectors)` | Query: find global coords by virtual coord criteria |
| `LookupAccessorConstructor` | xarray `.ulookup()` accessor (auto-constructs lookup) |

### TTL Sync (`data_conduit.ttlsync`)
| Function/Class | Purpose |
|----------------|---------|
| `extract_ttl_segments(times, values, ...)` | Binarize waveform → pulse table |
| `align_pulse_tables(ref_df, tgt_df, ...)` | Align two pulse tables for fitting |
| `fit_linear_timebase(ref_df, tgt_df, ...)` | Fit linear target→reference model |
| `convert_timebase(values, slope, intercept)` | Apply linear transform |
| `get_ttl_timebase_conversion(ref, tgt, ...)` | End-to-end conversion convenience |
| `get_npx_to_bonsai_time_conversion(...)` | NPX/Bonsai semantic wrapper |
| `TTLSyncModel(slope, intercept, r2)` | Dataclass with `.fit()` and `.transform()` |

### Global Times (`data_conduit.globaltimes`)
| Function | Purpose |
|----------|---------|
| `create_global_clock(start, end, step)` | Evenly spaced canonical time vector |
| `index_map_util(global_clock, stream_times, match_type)` | Map stream → global clock (nearest/before/after/exact) |

### Timestamps (`data_conduit.timestamps`)
| Function | Purpose |
|----------|---------|
| `collect_timestamps(df, ...)` | Extract timestamps from one DataFrame |
| `collect_timestamps_dict(dfs, ...)` | Extract from flat dict of DataFrames |
| `collect_timestamps_nested(dfs_dict, ...)` | Extract from arbitrarily nested dict |

### Segmentation (`data_conduit.segment`)
| Function | Purpose |
|----------|---------|
| `segment_boolean_series(series, ...)` | Run-length segmentation of boolean series |
| `slice_event_windows(df, event_times, pre, post)` | Cut DataFrame into event-centered windows |
| `slice_dataarray_windows(da, event_times, pre, post)` | Cut DataArray into event-centered windows |

### Utils (`data_conduit.utils`)
| Function | Purpose |
|----------|---------|
| `starts_with(prefix)` | Selector callable: key starts with prefix |
| `ends_with(suffix)` | Selector callable: key ends with suffix |
| `contains(substring)` | Selector callable: key contains substring |

### HarpTools (`data_conduit.harptools`)
| Function | Purpose |
|----------|---------|
| `read_harp_bin(path, harp_device_yaml_path)` | Read HARP .bin register file → DataFrame |
| `construct_device_reader(yaml_path)` | Build HARP reader from YAML schema |
| `collect_registers(yaml_path, addresses)` | Look up register names by address |

In [3]:
import inspect
import importlib
import re
import html as html_mod
from IPython.display import HTML


# ── Docstring parser (NumPy-style) ──────────────────────────────────────────

def _parse_numpy_docstring(doc: str) -> dict:
    """Parse a NumPy-style docstring into summary, parameters, and returns."""
    result = {"summary": "", "params": [], "returns": ""}
    if not doc:
        return result

    lines = doc.strip().splitlines()

    # Summary: everything before the first section header
    summary_lines = []
    i = 0
    while i < len(lines):
        if i + 1 < len(lines) and set(lines[i + 1].strip()) <= {"-"}  and lines[i + 1].strip():
            break
        summary_lines.append(lines[i].strip())
        i += 1
    result["summary"] = " ".join(summary_lines).strip()

    # Find sections
    sections = {}
    while i < len(lines):
        header = lines[i].strip()
        if (i + 1 < len(lines)
                and set(lines[i + 1].strip()) <= {"-"}
                and lines[i + 1].strip()):
            section_lines = []
            i += 2  # skip header + underline
            while i < len(lines):
                if (i + 1 < len(lines)
                        and set(lines[i + 1].strip()) <= {"-"}
                        and lines[i + 1].strip()):
                    break
                section_lines.append(lines[i])
                i += 1
            sections[header.lower()] = section_lines
        else:
            i += 1

    # Parse Parameters
    if "parameters" in sections:
        param_lines = sections["parameters"]
        j = 0
        while j < len(param_lines):
            line = param_lines[j]
            m = re.match(r"^(\w[\w\s,*]*?)\s*:\s*(.+)$", line)
            if m:
                pname, ptype = m.group(1).strip(), m.group(2).strip()
                desc_parts = []
                j += 1
                while j < len(param_lines) and (param_lines[j].startswith("    ") or param_lines[j].startswith("\t") or param_lines[j].strip() == ""):
                    stripped = param_lines[j].strip()
                    if stripped:
                        desc_parts.append(stripped)
                    j += 1
                result["params"].append((pname, ptype, " ".join(desc_parts)))
            else:
                j += 1

    # Parse Returns
    if "returns" in sections:
        ret_lines = [l.strip() for l in sections["returns"] if l.strip()]
        result["returns"] = " ".join(ret_lines)

    return result


# ── HTML renderer ────────────────────────────────────────────────────────────

def _render_module_section(module, title: str) -> str:
    """Render one module's API as a styled HTML section."""
    items_html = []

    for name in sorted(module.__all__):
        obj = getattr(module, name)
        is_class = inspect.isclass(obj)
        badge = ("cls", "#6f42c1") if is_class else ("fn", "#0969da")

        try:
            sig = str(inspect.signature(obj))
        except (ValueError, TypeError):
            sig = "(...)"

        doc = inspect.getdoc(obj)
        parsed = _parse_numpy_docstring(doc)
        summary = html_mod.escape(parsed["summary"]) or "<em>No description</em>"

        # Parameter rows
        param_rows = ""
        if parsed["params"]:
            param_rows = "".join(
                f'<tr style="border-bottom:1px solid #eee;">'
                f'<td style="padding:4px 10px;font-family:monospace;white-space:nowrap;color:#24292f;"><strong>{html_mod.escape(pn)}</strong></td>'
                f'<td style="padding:4px 10px;font-family:monospace;color:#6a737d;font-size:0.85em;">{html_mod.escape(pt)}</td>'
                f'<td style="padding:4px 10px;color:#57606a;font-size:0.88em;">{html_mod.escape(pd)}</td>'
                f"</tr>"
                for pn, pt, pd in parsed["params"]
            )

        # Build the collapsible card
        detail_block = ""
        if param_rows:
            detail_block = (
                f'<details style="margin-top:6px;">'
                f'<summary style="cursor:pointer;font-size:0.85em;color:#0969da;font-weight:500;">Parameters</summary>'
                f'<table style="margin:6px 0 2px 0;border-collapse:collapse;width:100%;">'
                f'<tr style="border-bottom:2px solid #d0d7de;text-align:left;">'
                f'<th style="padding:4px 10px;font-size:0.8em;color:#656d76;">Name</th>'
                f'<th style="padding:4px 10px;font-size:0.8em;color:#656d76;">Type</th>'
                f'<th style="padding:4px 10px;font-size:0.8em;color:#656d76;">Description</th>'
                f'</tr>{param_rows}</table></details>'
            )

        sig_escaped = html_mod.escape(sig)
        items_html.append(
            f'<div style="border:1px solid #d0d7de;border-radius:8px;padding:12px 16px;margin-bottom:8px;background:#fff;">'
            f'<div style="display:flex;align-items:baseline;gap:8px;">'
            f'<span style="background:{badge[1]};color:#fff;font-size:0.7em;padding:2px 7px;border-radius:4px;'
            f'font-weight:600;letter-spacing:0.5px;text-transform:uppercase;">{badge[0]}</span>'
            f'<code style="font-size:0.95em;font-weight:600;color:#24292f;">{html_mod.escape(name)}</code>'
            f'<code style="font-size:0.82em;color:#6a737d;">{sig_escaped}</code>'
            f'</div>'
            f'<div style="margin-top:4px;color:#57606a;font-size:0.9em;">{summary}</div>'
            f'{detail_block}'
            f'</div>'
        )

    return (
        f'<div style="margin-bottom:24px;">'
        f'<h3 style="border-bottom:2px solid #d0d7de;padding-bottom:6px;margin-bottom:12px;">{title}</h3>'
        f'{"".join(items_html)}'
        f'</div>'
    )


# ── Build the full reference ─────────────────────────────────────────────────

module_specs = {
    "IO": "data_conduit.io",
    "DataSource": "data_conduit.datasource",
    "MultiSource": "data_conduit.multisource",
    "Virtual Arrays": "data_conduit.virtualarrays",
    "TTL Sync": "data_conduit.ttlsync",
    "Global Times": "data_conduit.globaltimes",
    "Segmentation": "data_conduit.segment",
    "Timestamps": "data_conduit.timestamps",
    "Utils": "data_conduit.utils",
    "HarpTools": "data_conduit.harptools",
}

sections = []
for title, mod_path in module_specs.items():
    try:
        mod = importlib.import_module(mod_path)
        sections.append(_render_module_section(mod, title))
    except Exception as e:
        sections.append(
            f'<div style="margin-bottom:24px;">'
            f'<h3 style="border-bottom:2px solid #d0d7de;padding-bottom:6px;">{html_mod.escape(title)}</h3>'
            f'<p style="color:#cf222e;font-style:italic;">Skipped — import failed: {html_mod.escape(str(e))}</p>'
            f'</div>'
        )

header_html = (
    '<div style="margin-bottom:16px;">'
    '<h2 style="margin-bottom:4px;">API Quick-Reference</h2>'
    '<p style="color:#57606a;font-size:0.9em;margin-top:0;">'
    'Auto-generated from source docstrings. Click <strong>Parameters</strong> to expand details.</p>'
    '</div>'
)

display(HTML(header_html + "\n".join(sections)))

Name,Type,Description
base_path,str or Path,Root directory to walk.
readers,"dict[str, Callable] or None","Mapping of file extensions to reader functions. Each reader must follow: (file_path, **kwargs) -> pd.DataFrame. e.g. {'.bin': read_harp_bin, '.csv': read_csv} If None, all files are stored as Paths."
reader_kwargs,"dict[str, dict] or None","Mapping of file extensions to kwargs dicts for each reader. e.g. {'.bin': {'harp_reader': r, 'addr_to_name': m}} Extensions not in this dict receive no extra kwargs."
keep_empty,bool,"If True, preserve empty subdirectories as empty dicts. Default is True."
flatten,bool,"If True, return a flat dict with keys joined by separator."
separator,str,Separator for flattened keys. Default is ':'.
verbose,bool,"If True, prints warnings during processing."
Name,Type,Description
base_path,str or Path,The root directory to walk.
file_pattern,str,"Glob pattern for matching files (e.g. '*.csv', '*.bin', '*'). Only applied to files, not directories."
